# 06 — Shift-corrected OTCP and GRCP baselines (Parkinson's)
Reviewers will ask: "weighted conformal prediction works with any scalar score — why not simply weight
OTCP or GRCP?" This notebook adds exactly those two baselines:

* **OTCP + WCP** — the OTCP score of Klein et al. (2026), calibrated with the weighted quantile of
  Tibshirani et al. (2019) using the estimated density ratio.
* **GRCP + WCP** — the same for the spatial-rank score of Conrad et al. (2026).

**[my choice]** Only the calibration step is weighted; the OT map and the rank estimator are still
fitted on the unweighted hold-out data (weighting the map itself is an open question).

Everything else is identical to notebook 04 (same seed, splits, shift and settings), so the nine
original methods should reproduce notebook 04's numbers exactly — the last cell checks this.

**Weight clipping (`RATIO_CLIP`).** After notebook 08, the main results use density-ratio weights capped
at the 99% quantile of the training weights (`RATIO_CLIP = 0.99`) for ALL weighted methods
**[my choice]**. Results then go to `results/weighted_baselines_clip99/`. `RATIO_CLIP = None` gives the
original unclipped run in `results/weighted_baselines/` (and the check against notebook 04).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import cpshift as cp
import experiment as ex
import deep

print("device:", deep.DEVICE)

## Configuration (identical to notebook 04)

In [ ]:
QUICK = True
RATIO_CLIP = 0.99      # None = no clipping (original run)

CFG = dict(
    SEED=2026, ALPHA=0.10, N_TRAIN=1000, N_TEST=1000, N_REPS=20, B=1000, SCP_FIT_FRAC=0.5,
    K_VALUES=[0, 1, 2, 3, 4, 5, 6, 7],
    DEEP_CAL_FRAC=0.5, DEEP_VAL_FRAC=0.2, HIDDEN=64, N_LAYERS=2, LR=1e-3, WEIGHT_DECAY=1e-4,
    BATCH=64, MAX_EPOCHS=500, PATIENCE=30, EPS=1e-3,
    OT_M=1024, OT_EPS=0.1, GRID_SIDE=150, GRCP_VOL_POINTS=100, RATIO_CLIP=RATIO_CLIP,
)
CC = dict(WSC_DELTA=0.10, WSC_N_DIR=1000, WSC_GRID=50, CEC_J=10)
if QUICK:
    CFG.update(N_REPS=1, B=200, MAX_EPOCHS=40, K_VALUES=[0, 4, 7], GRCP_VOL_POINTS=20)
    CC.update(WSC_N_DIR=100)

OUT_DIR = "../results/weighted_baselines" + (f"_clip{round(100 * RATIO_CLIP)}" if RATIO_CLIP else "")
os.makedirs(OUT_DIR, exist_ok=True)
CFG

## Run

In [ ]:
df, X_all, Y_all = cp.load_parkinsons("../data/parkinsons_updrs.data")
res_df = ex.run(X_all, Y_all, cp.ShiftSampler(X_all), CFG, CC, deep, methods=ex.METHODS_ALL)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Tables

In [ ]:
by_k, tables = ex.summarize(res_df, order=ex.METHODS_ALL)
by_k.to_csv(f"{OUT_DIR}/summary_by_k.csv", index=False)
main_k = [0, 4, 7]
compact = pd.concat({f"k={k}": by_k[by_k.k == k].set_index('method')[['coverage', 'volume', 'WSC', 'pct_infinite']]
                     for k in main_k if k in CFG['K_VALUES']}, axis=1).reindex(ex.METHODS_ALL)
compact.to_csv(f"{OUT_DIR}/table_main.csv")
print("MAIN TABLE\n", compact.to_string())

## Paired comparisons at the strongest shift
Same splits for every method, so differences are paired over splits.

In [ ]:
from scipy import stats
kmax = max(CFG['K_VALUES'])
d = res_df[res_df.k == kmax].pivot(index='rep', columns='method', values=['coverage', 'volume_median', 'WSC'])
pairs = [('Deep-LocalCov + WCP (ours)', 'OTCP + WCP'), ('Deep-LocalCov + WCP (ours)', 'GRCP + WCP'),
         ('Deep-LocalCov + EUc (ours)', 'OTCP + WCP'), ('Deep-LocalCov + EUc (ours)', 'GRCP + WCP')]
rows = []
for m1, m2 in pairs:
    for metric in ['coverage', 'volume_median', 'WSC']:
        x = d[metric][m1] - d[metric][m2]
        x = x[np.isfinite(x)]
        p = stats.ttest_1samp(x, 0).pvalue if len(x) > 1 else np.nan
        rows.append(dict(comparison=f"{m1} - {m2}", metric=metric, mean_diff=x.mean(), n=len(x), p_t=p))
paired = pd.DataFrame(rows).round(4)
paired.to_csv(f"{OUT_DIR}/paired_k{kmax}.csv", index=False)
print(paired.to_string())

## Figure

In [ ]:
style = {'OTCP': ('tab:purple', 'v', '--'), 'OTCP + WCP': ('tab:purple', 'v', '-'),
         'GRCP': ('tab:olive', 'P', '--'), 'GRCP + WCP': ('tab:olive', 'P', '-'),
         'EUc-LR': ('tab:red', 'o', '-'),
         'Deep-LocalCov + WCP (ours)': ('tab:green', '^', '-'),
         'Deep-LocalCov + EUc (ours)': ('tab:blue', 'D', '-')}
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for m, (c, mk, ls) in style.items():
    s = by_k[by_k.method == m]
    ax[0].plot(s.k, s.coverage, color=c, marker=mk, ls=ls, label=m)
    ax[1].plot(s.k, s.volume, color=c, marker=mk, ls=ls)
    ax[2].plot(s.k, s.WSC, color=c, marker=mk, ls=ls)
ax[0].axhline(90, color='k', lw=1, ls=':')
ax[0].set_ylabel('coverage (%)'); ax[1].set_ylabel('region area (UPDRS$^2$)'); ax[2].set_ylabel('WSC (%)')
ax[1].set_yscale('log')
for x in ax:
    x.set_xlabel('shift intensity $k$')
fig.legend(loc='lower center', ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_weighted_baselines.png", dpi=200, bbox_inches='tight'); plt.show()

## Check: the nine original methods must match notebook 04
Only meaningful after the full run (QUICK = False) and if notebook 04's full results are present.

In [ ]:
ref_path = "../results/benchmark_parkinsons/summary_by_k.csv"
if os.path.exists(ref_path) and not QUICK and not RATIO_CLIP:
    ref = pd.read_csv(ref_path).set_index(['method', 'k'])
    new = by_k.set_index(['method', 'k']).loc[ref.index]
    diff = (new[['coverage', 'volume', 'WSC']] - ref[['coverage', 'volume', 'WSC']]).abs().max()
    print("max |difference| vs notebook 04:\n", diff, "\n->", "IDENTICAL" if diff.max() < 1e-6 else "DIFFERENT — tell Claude")
else:
    print("skipped (QUICK run, clipped weights, or notebook 04 results not found)")

In [ ]:
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))
print("CONFIG:", CFG)